# 🏆 Capstone Step 6 — Architecture Diagram + Cost Analysis + Executive Summary
**Prerequisite:** Complete `lesson_09e_eval_monitoring.ipynb` first

In [ ]:
# ── Capstone imports & env check ──────────────────────────────────
import os, json, time, datetime, re, asyncio
from pathlib import Path
from typing import Optional, List, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field as dc_field
import pandas as pd
import numpy as np

from openai import OpenAI
from anthropic import Anthropic
from datasets import load_dataset
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Check M8 artifacts exist
ARTIFACTS = {
    "complaints_extracted": Path("data/capstone/complaints_extracted.jsonl"),
    "hybrid_routing_log":   Path("data/capstone/hybrid_routing_log.jsonl"),
    "golden_eval":          Path("data/capstone/golden_eval.jsonl"),
}
for name, path in ARTIFACTS.items():
    status = "✅" if path.exists() else "⚠️  MISSING — run Module 8 first"
    print(f"  {status}  {name}: {path}")

print("\n✅ Capstone imports ready")

---
## 🔴 CHALLENGE Step 6 — Architecture Diagram + Cost Analysis + Executive Summary

**Deliverables:**
1. Architecture diagram (Mermaid — rendered below)
2. Cost for 100 customers + 10k/month projection
3. 1-page jargon-free executive summary

**Auto-check verifies:**
- ✓ Architecture diagram exists and is readable
- ✓ Total cost for 100 customers computed
- ✓ Projection for 10,000 customers/month with recommendation
- ✓ Executive summary is jargon-free and ≤ 500 words


In [ ]:
show_exercise(
    "CAP-6", "Architecture + Cost + Executive Summary",
    "Draw architecture diagram (Mermaid/ASCII). Compute total cost. Write 1-page executive summary.",
    hints=[
        "▸ Cost: sum all LLM API calls (input + output tokens × pricing) + $0 for LM Studio",
        "▸ Executive summary: no jargon, focus on business value ('reduces analyst time by...')",
        "▸ Mermaid: use flowchart LR for left-right data flow",
    ],
    checks=[
        "Architecture diagram exists and is readable",
        "Total cost for 100 customers computed",
        "Projection for 10,000/month with recommendation",
        "Executive summary ≤ 500 words, jargon-free"
    ],
    exercise_type="CHALLENGE"
)

In [ ]:
# ── Architecture diagram (Mermaid) ────────────────────────────────
architecture_mermaid = """
flowchart LR
    subgraph INGESTION["🗄️ Ingestion Layer (M1+M2)"]
        A[(CFPB\nComplaints)] --> B[trim_to_budget\n≤2000 tokens]
        B --> C[ComplaintRecord\nPydantic]
    end
    subgraph INTEL["🧠 Intelligence Core (M3+M5)"]
        D[(ChromaDB\n30 Policies)] --> E[PolicyRAGNode]
        F[XGBoost\nChurn Model] --> G[ChurnAnalysisNode]
        E & G --> H{hybrid_route}
    end
    subgraph AGENTS["🤖 Agent Orchestrator (M4+M7)"]
        I[OrchestratorNode] --> E & G
        E & G --> J[RetentionOfferNode]
        J --> K{Guardrail\nchurn>0.6?}
        K -->|yes| L[Draft Offer]
        K -->|no| M[Block + Log]
        L -->|value>100PLN| N[HITL Interrupt]
        L --> O[AuditNode]
        M --> O
    end
    subgraph OUTPUT["📋 Output & Monitoring (M2+M6+M7)"]
        O --> P[CRMIntelligenceReport\nPydantic]
        P --> Q[Langfuse\nTracing]
        P --> R[RAGAS\nEval]
        P --> S[LLM-as-Judge]
    end
    C --> I
    style INGESTION fill:#e8f4f8,stroke:#2196F3
    style INTEL     fill:#f3e8f8,stroke:#9C27B0
    style AGENTS    fill:#f8f3e8,stroke:#FF9800
    style OUTPUT    fill:#e8f8e8,stroke:#4CAF50
"""

print("Mermaid architecture diagram:")
print(architecture_mermaid)
# In Jupyter: display with %%capture or use mermaid extension

In [ ]:
# ── Cost analysis ─────────────────────────────────────────────────
n_reports = len(ALL_REPORTS)
total_cost = sum(r.total_cost_usd for r in ALL_REPORTS)
avg_cost_per_customer = total_cost / max(n_reports, 1)
monthly_10k_cost = avg_cost_per_customer * 10_000

print("💰 Cost Analysis")
print("=" * 50)
print(f"  Customers processed:         {n_reports}")
print(f"  Total cost (100 customers):  ${total_cost:.4f}")
print(f"  Avg cost per customer:       ${avg_cost_per_customer:.6f}")
print(f"  LM Studio local calls:       $0.00 (free)")
print(f"  Cloud API overhead:          ${total_cost:.4f}")
print()
print(f"  📈 Projection (10,000/month): ${monthly_10k_cost:.2f}/month")
print()
if monthly_10k_cost < 50:
    print("  ✅ RECOMMENDATION: Highly cost-effective at scale.")
    print("     Consider upgrading to GPT-4o for complex cases only (+$X/month).")
elif monthly_10k_cost < 200:
    print("  ✅ RECOMMENDATION: Cost-effective. Monitor usage and tune model mix.")
else:
    print("  ⚠️  RECOMMENDATION: Optimize model mix — increase LM Studio usage for routine cases.")

In [ ]:
# ── Executive summary (jargon-free) ─────────────────────────────-─
exec_summary = f"""# Banking CRM Intelligence System — Executive Summary

**What it does:**
This system automatically processes customer complaints and identifies which customers are at risk
of leaving the bank. For each customer, it reads their complaint, checks relevant bank policies,
assesses their likelihood to cancel, and — where appropriate — drafts a personalized retention offer.
Everything is logged for compliance review.

**Business value:**
- Processes {n_reports} customer cases in under {avg_processing_ms:.0f} ms average per customer
- Identifies high-risk customers before they leave, enabling proactive outreach
- Retention offers are only sent to customers most likely to churn (>{60}% probability),
  avoiding unnecessary discounts for customers who were going to stay anyway
- All offers above PLN 100 in value require a human advisor's approval before sending
- Every decision is logged with a reason, supporting regulatory audit requirements

**Cost:**
Running this system for 100 customers costs approximately ${total_cost:.4f}.
Scaled to 10,000 customers per month, the estimated cost is **${monthly_10k_cost:.2f}/month** —
the equivalent of less than one hour of analyst time.

**Safeguards:**
- No real customer documents are sent to external AI services (GDPR-compliant)
- Low-risk customers are automatically excluded from retention offers
- High-value offers require human review
- Every action is timestamped and auditable

**Next steps:**
1. Pilot with 500 real customers (anonymised complaints)
2. Connect to live CRM system for automated outreach
3. Review retention offer effectiveness after 90 days

*Prepared by: LLM Engineering Team | {datetime.datetime.now().strftime('%B %Y')}*
"""

word_count = len(exec_summary.split())
print(exec_summary)
print(f"\nWord count: {word_count} (limit: 500)")

Path("executive_summary.md").write_text(exec_summary)
print("\n✅ Executive summary saved → executive_summary.md")

In [ ]:
# ── Auto-check Step 6 ─────────────────────────────────────────────
exec_word_count = len(exec_summary.split())

check([
    (len(architecture_mermaid.strip()) > 200,    "Architecture diagram has content (> 200 chars)"),
    (total_cost >= 0,                             "Total cost computed"),
    (monthly_10k_cost > 0,                        "10k/month projection computed"),
    (exec_word_count <= 500,                      f"Executive summary ≤ 500 words (got {exec_word_count})"),
    (Path("executive_summary.md").exists(),       "executive_summary.md written"),
], exercise_id="CAP-6")

---
## 🏁 Final Progress Check


In [ ]:
progress()

In [ ]:
# ── Full capstone checklist ────────────────────────────────────────
print("\n" + "═"*60)
print("  🏆  CAPSTONE FINAL CHECKLIST")
print("═"*60)

checks = [
    (len(ALL_REPORTS) >= 90,          "100 CRMIntelligenceReport objects generated"),
    (INGESTION_PASS_RATE >= 0.90,     "Pydantic extraction pass rate ≥ 90%"),
    (policy_collection.count() >= 30, "ChromaDB with ≥ 30 policies indexed"),
    (AUC >= 0.68,                     "XGBoost AUC ≥ 0.68"),
    (Path("data/capstone/hybrid_routing_log.jsonl").exists(), "hybrid_routing_log.jsonl saved"),
    (len(HITL_TRIGGERED) >= 0,        "HITL trigger logic implemented"),
    (len(ragas_scores) >= 10,         "RAGAS eval on ≥ 10 policy Q&A pairs"),
    (avg_judge >= 3.0,                f"LLM-as-judge avg ≥ 3.0/5.0 (got {avg_judge:.2f})"),
    (Path("capstone_eval_report.md").exists(), "capstone_eval_report.md written"),
    (Path("executive_summary.md").exists(),    "Executive summary written"),
    (exec_word_count <= 500,          f"Executive summary ≤ 500 words"),
    (total_cost >= 0,                 "Pipeline cost computed"),
]
n_pass = sum(1 for ok, _ in checks if ok)
for ok, label in checks:
    print(f"  {'✅' if ok else '❌'}  {label}")
print("═"*60)
print(f"  Score: {n_pass}/{len(checks)}")
if n_pass == len(checks):
    print("\n  🎉 CONGRATULATIONS! You have completed the Capstone!")
    print("     These are Senior LLM Engineer skills. Push to GitHub.")
else:
    print(f"\n  {len(checks)-n_pass} items remaining. Keep going!")
print("═"*60)

---
## 🎉 Congratulations!

Completing this Capstone means you can build **production LLM systems** that integrate:

| Skill | Module |
|-------|--------|
| Prompt engineering + context management | M1 |
| Structured output with Pydantic + Instructor | M2 |
| RAG with ChromaDB + LangChain | M3 |
| LangGraph agents with HITL | M4 |
| Hybrid ML + LLM pipelines | M5 |
| RAGAS + LLM-as-judge + Langfuse | M6 |
| Reasoning models + Vision + Multi-agent | M7 |

**These skills separate a Senior LLM Engineer from a junior developer in 2026.**

---
### 📣 Next Steps
1. `git push` — add this to your GitHub portfolio
2. Write a LinkedIn post with the architecture diagram
3. Consider contributing to an open-source LLM project
4. Follow: **Anthropic Research · DeepMind · Hugging Face Blog** for what comes next

*Good luck! 🚀*
